# EDA into yfinance

In [2]:
# Cell 1 — Pull raw data from yfinance for one ticker
import yfinance as yf
import pandas as pd

raw = yf.download("NVDA", period="1y", auto_adjust=True)
raw = raw.reset_index()
print(raw.shape)      # How many rows and columns?
raw.head(10)          # What does the data actually look like?

[*********************100%***********************]  1 of 1 completed

(251, 6)


Price,Date,Close,High,Low,Open,Volume
Ticker,,NVDA,NVDA,NVDA,NVDA,NVDA
0,2025-03-13,115.552773,117.732259,113.763193,117.002428,299033100
1,2025-03-14,121.641335,121.851284,118.122167,118.582058,277593500
2,2025-03-17,119.501839,122.861048,118.002192,122.711082,255501500
3,2025-03-18,115.402809,118.991960,114.513019,117.972203,299686900
4,2025-03-19,117.492317,120.421627,115.652754,117.242376,273426200
5,2025-03-20,118.502068,120.171672,116.442555,116.522538,248829700
6,2025-03-21,117.672264,117.962197,115.392803,116.912449,266498500
7,2025-03-24,121.381401,122.191208,119.311881,119.851755,228452500
8,2025-03-25,120.661568,121.261425,118.891980,120.521601,167447200


In [5]:
# Cell 2 — Inspect the data types of each column
# This is critical for Silver — you need to know what types
# yfinance actually returns vs what you need them to be
raw.dtypes

Price   Ticker
Date              datetime64[ns]
Close   NVDA             float64
High    NVDA             float64
Low     NVDA             float64
Open    NVDA             float64
Volume  NVDA               int64
dtype: object

In [6]:
# Cell 3 — Check for nulls across every column
raw.isnull().sum()

Price   Ticker
Date              0
Close   NVDA      0
High    NVDA      0
Low     NVDA      0
Open    NVDA      0
Volume  NVDA      0
dtype: int64

In [7]:
# Cell 4 — Look at basic statistics to spot anomalies
# Are there any suspiciously low prices? Any zeros?
raw.describe()

Price,Date,Close,High,Low,Open,Volume
Ticker,,NVDA,NVDA,NVDA,NVDA,NVDA
count,251,251.000000,251.000000,251.000000,251.000000,2.510000e+02
mean,2025-09-10 18:10:02.390438144,164.475138,166.757950,161.922467,164.443011,2.005099e+08
min,2025-03-13 00:00:00,94.287781,99.416579,86.599596,87.439395,6.552850e+07
25%,2025-06-11 12:00:00,143.316177,144.860891,141.836424,142.566394,1.540318e+08
50%,2025-09-11 00:00:00,177.800476,180.170056,175.020530,177.750479,1.821366e+08
75%,2025-12-09 12:00:00,184.944992,187.269434,182.185134,184.759707,2.265876e+08
max,2026-03-12 00:00:00,207.017273,212.166717,205.537422,208.057150,6.129183e+08
std,NaN,28.873243,29.175938,28.963819,29.273976,7.385781e+07


In [8]:
# Cell 5 — Now read from your actual Bronze Parquet file
# This is more important than raw yfinance because Silver
# reads from Bronze, not directly from yfinance
import duckdb

conn = duckdb.connect()
df = conn.execute("SELECT * FROM '../data/bronze/*.parquet'").df()
print(df.shape)
print(df.dtypes)
df.head(10)

(2271, 11)
Date             datetime64[ns]
Close                   float64
High                    float64
Low                     float64
Open                    float64
Volume                    int64
ticker_symbol            object
ingested_at              object
source_system            object
batch_id                 object
ingested_date            object
dtype: object


,Date,Close,High,Low,Open,Volume,ticker_symbol,ingested_at,source_system,batch_id,ingested_date
0,2025-03-10,513.121155,513.121155,513.121155,513.121155,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
1,2025-03-11,509.238556,509.238556,509.238556,509.238556,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
2,2025-03-12,511.738007,511.738007,511.738007,511.738007,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
3,2025-03-13,504.674194,504.674194,504.674194,504.674194,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
4,2025-03-14,515.512024,515.512024,515.512024,515.512024,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
5,2025-03-17,518.851257,518.851257,518.851257,518.851257,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
6,2025-03-18,513.338501,513.338501,513.338501,513.338501,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
7,2025-03-19,518.890747,518.890747,518.890747,518.890747,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
8,2025-03-20,517.813904,517.813904,517.813904,517.813904,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
9,2025-03-21,518.258545,518.258545,518.258545,518.258545,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09


In [9]:
# Cell 6 — Check what columns came through in the Parquet file
# Sometimes Parquet serialization changes things slightly
df.columns.tolist()

['Date',
 'Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'ticker_symbol',
 'ingested_at',
 'source_system',
 'batch_id',
 'ingested_date']

In [10]:
print('done')

done
